In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from src.evaluation import poisson_deviance, calibration_by_exposure
from src import gbm

df = pd.read_csv("../data/processed/policies_clean.csv")
grp = pd.read_csv("../data/processed/policy_groups.csv")
tr_ids = pd.read_csv("../data/processed/train_ids.csv")["IDpol"]
te_ids = pd.read_csv("../data/processed/test_ids.csv")["IDpol"]
assert set(tr_ids).isdisjoint(te_ids)

train = df[df["IDpol"].isin(tr_ids)].reset_index(drop=True)
test = df[df["IDpol"].isin(te_ids)].merge(grp, on="IDpol").reset_index(drop=True)
assert len(train) == len(tr_ids) and len(test) == len(te_ids)

expo = test["Exposure"].to_numpy()
y = test["ClaimNb"].to_numpy(float)

rate = train["ClaimNb"].sum() / train["Exposure"].sum()
mu_base = rate * expo

m1, lv1 = gbm.fit(train, gbm.PARAMS, gbm.FEATURES_V1)
mu_v1 = m1.predict(gbm.make_X(test, lv1, gbm.FEATURES_V1)) * expo
m2, lv2 = gbm.fit(train, gbm.PARAMS, gbm.FEATURES_V2)
mu_v2 = m2.predict(gbm.make_X(test, lv2, gbm.FEATURES_V2)) * expo

for name, mu in [("baseline", mu_base), ("GBM v1", mu_v1), ("GBM v2", mu_v2)]:
    print(f"{name:9s} test deviance: {poisson_deviance(y, mu):.4f} | actual/predicted overall: {y.sum()/mu.sum():.4f}")

# uncertainty: bootstrap over feature-profile groups (twins stay together)
def unit_dev(y, mu):
    t = np.zeros_like(y); p = y > 0
    t[p] = y[p] * np.log(y[p] / mu[p])
    return 2 * (t - (y - mu))
d_base, d_v2 = unit_dev(y, mu_base), unit_dev(y, mu_v2)
g = test.groupby("Group").indices
keys = np.array(list(g.keys()))
sums = pd.DataFrame({"b": [d_base[g[k]].sum() for k in keys],
                     "v": [d_v2[g[k]].sum() for k in keys],
                     "n": [len(g[k]) for k in keys]})
rng = np.random.default_rng(0)
gains = []
for _ in range(500):
    s = sums.iloc[rng.integers(0, len(sums), len(sums))]
    gains.append((s["b"].sum() - s["v"].sum()) / s["n"].sum())
print("v2 gain over baseline, 95% bootstrap interval:", np.percentile(gains, [2.5, 97.5]).round(4))

print("\nCalibration by exposure band (v2):")
print(calibration_by_exposure(test, mu_v2), "\n")

age0 = (test["VehAge"] == 0).to_numpy()
rows = []
for thr in [0.75, 0.9]:
    for name, gmask in [("age 0", age0), ("age 1+", ~age0)]:
        s = (test["Exposure"].to_numpy() >= thr) & gmask
        rows.append({"exposure >=": thr, "group": name, "claims": int(y[s].sum()),
                     "v1 actual/pred": round(y[s].sum() / mu_v1[s].sum(), 3),
                     "v2 actual/pred": round(y[s].sum() / mu_v2[s].sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

baseline  test deviance: 0.3307 | actual/predicted overall: 1.0026
GBM v1    test deviance: 0.3025 | actual/predicted overall: 1.0013
GBM v2    test deviance: 0.2892 | actual/predicted overall: 1.0034
v2 gain over baseline, 95% bootstrap interval: [0.0386 0.0449]

Calibration by exposure band (v2):
             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]      25542     675      683.6             0.987
(0.1, 0.25]     19162     781      727.9             1.073
(0.25, 0.5]     26194    1269     1298.6             0.977
(0.5, 0.75]     18527    1146     1187.9             0.965
(0.75, 1.0]     45932    3345     3293.5             1.016 

 exposure >=  group  claims  v1 actual/pred  v2 actual/pred
        0.75  age 0     133           0.534           1.182
        0.75 age 1+    3318           0.861           1.014
        0.90  age 0      30           0.419           1.008
        0.90 age 1+    2725           0.842  

In [2]:
import joblib
from pathlib import Path
Path("../models").mkdir(exist_ok=True)
joblib.dump({"model": m2, "levels": lv2, "version": "gbm_v2",
                "features": gbm.FEATURES_V2, "predict_at_exposure": 1.0,
                "portfolio_rate": rate},
               "../models/gbm_v2.joblib")

['../models/gbm_v2.joblib']

In [3]:
import sys; sys.path.append("..")
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.evaluation import poisson_deviance
from src import gbm

bundle = joblib.load("../models/gbm_v2.joblib")
print({k: v for k, v in bundle.items() if k not in ("model", "levels")})
print("lightgbm now:", lgb.__version__)

df = pd.read_csv("../data/processed/policies_clean.csv")
te_ids = pd.read_csv("../data/processed/test_ids.csv")["IDpol"]
test = df[df["IDpol"].isin(te_ids)].reset_index(drop=True)

X = gbm.make_X(test, bundle["levels"], bundle["features"])
mu = bundle["model"].predict(X) * test["Exposure"].to_numpy()
dev = poisson_deviance(test["ClaimNb"], mu)
print("reloaded model, test deviance:", round(dev, 4))
assert abs(dev - 0.2892) < 5e-4, "saved model does not reproduce the evaluated one"

# the quoting setting: annual rate at Exposure = 1
rate1 = bundle["model"].predict(gbm.make_X(test.assign(Exposure=1.0), bundle["levels"], bundle["features"]))
print("annual rate at Exposure=1: mean", rate1.mean().round(4), "| min", rate1.min().round(4), "| max", rate1.max().round(4))

{'version': 'gbm_v2', 'features': ['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density', 'VehBrand', 'VehGas', 'Region', 'bm_off_ladder', 'Exposure'], 'predict_at_exposure': 1.0, 'portfolio_rate': np.float64(0.10088360486835339)}
lightgbm now: 4.7.0
reloaded model, test deviance: 0.2892
annual rate at Exposure=1: mean 0.0809 | min 0.0121 | max 0.9604
